# FourFusion · FLAG 2027 · reproduction of the Evaluation submission 13.54 · FF_MODE=train

Team **FourFusion** (CodaBench `minhnhien2811`): Nguyen Chon Nhan, Dang Minh Nhien, Vu Nguyen Anh Dung, Nguyen Huu Bao Loc;
advisor Nguyen Duy Tam Anh; University of Information Technology, VNU-HCM.

Target: the displayed Evaluation submission (CodaBench submission 1054704, EER 13.54 = 10.20 · 11.97 · 12.98 · 18.99), file
`xjf_r10_trsig.zip` (SHA-256 `b4354eaf988b2b4ea8c4e1c3fa44d7bcf631ea5cb3cd635ab2965021fe82e507`). This notebook runs
**mode (a): rebuild everything from the raw and collected data (about 150-170 GPU-hours; split over many sessions with FF_ONLY)**. Accelerator: GPU (T4, some stages two GPUs) needed.

| file of the zip | value (stored as −score: lower = same person) |
|---|---|
| no_gender / Bangla | −(logit F1s′ + logit RFX) |
| gender / Bangla | −(logit F1s′ + logit RFX′), RFX′ = RFX with the FVBIND B2 voice block |
| both English files | (FACTOR_ENFUSE English value) / σ[protocol] − 1.0 × z of the DP (Different Profile) branch; σ = 5.0983 (no_gender), 3.8376 (gender) |

How it runs: every original Kaggle run is a **stage** of the repository (`stages/<group>/<id>/stage.json` + `stage.ipy`, original
code cells verbatim with the listed changes). `runner/ffrun.py` plans the stages needed for the target, runs each in its own
process with a private `/kaggle/input` (the inputs the original run saw) and `/kaggle/working`, and keeps what a stage leaves as
an **artifact** under `FF_WORK/art/<original Kaggle id>`. An artifact comes from: this run → a saved copy (attached datasets,
matched by slug; `FF_DATA` folders) → the stage that makes it.

Attach: (1) the code bundle (`python runner/build_kaggle_notebook.py --bundle <dir>`, uploaded as a private dataset) unless
`FF_REPO_GIT` is set; (2) the data the plan lists as missing (cell 03 prints them): original Kaggle datasets are found by slug,
repository ids (`raw/...`, `scores/...`, `pretrained/...`) and the flat `05_scores/` files only under an `FF_DATA` root: with the
default `FF_DATA=auto` every attached folder that holds `00_raw/`, `01_collected/`, ..., `06_pretrained/` (the Drive layout) is used
(cell 02 prints what was found). **Mode (c) needs the folder holding `05_scores/` attached.** (3) For a continued session, the output
of the previous session of this notebook. Secrets: add `HF_TOKEN` as a Kaggle secret if a stage needs it (read, never printed).

**Literal `/kaggle/input` stages.** Some stages run code decoded from embedded payloads that scans the literal `/kaggle/input`
(the runner marks them `[literal /kaggle/input]` in the plan: e50cc, e112f0sx, e112f1sx, the F family, the RFX projector runs,
t189_b2 / t192_fvbx / t163p_fvb). In a Kaggle session that code sees every attachment of the session and none of the artifacts made
earlier in the same session. With `FF_LITERAL_STRICT=1` (default here) the runner therefore refuses such a stage unless every input
is attached at `/kaggle/input` and nothing else is (except this code bundle). Run each of them alone: a new session, `FF_ONLY=<stage>`,
exactly its inputs attached as Kaggle datasets / kernel outputs (plus the previous session's output for inputs made there). Mode (b)
on Kaggle therefore needs separate sessions for e50cc, e112f0sx, e112f1sx and t192_fvbx; on Linux or in a container the runner points
`/kaggle/input` at each stage's own inputs and one session runs the whole plan (`runner/README.md`).

In [ ]:
print("[cell] 00 config", flush=True)
print("[boot] FourFusion 13.54 orchestrator, FF_MODE=train", flush=True)
import os
os.environ.setdefault("FF_MODE", "train")              # infer | train | assemble
os.environ.setdefault("FF_TARGET", "final_verify")      # stage id(s) or artifact id(s)
os.environ.setdefault("FF_USE_SAVED", "collected")        # all | collected | none
os.environ.setdefault("FF_ONLY", "")                    # e.g. "t153c" to run one stage in this session
os.environ.setdefault("FF_FORCE", "")
os.environ.setdefault("FF_SMOKE", "0")                  # "1" = smoke parameters of every stage (CPU allowed)
os.environ.setdefault("FF_PIP", "original")             # original | pinned | skip
os.environ.setdefault("FF_TIME_BUDGET_H", "11.3")
os.environ.setdefault("FF_WORK", "/kaggle/working/ff_work")
os.environ.setdefault("FF_DATA", "auto")                # auto = every attached folder holding 00_raw/ ... 06_pretrained/ (Drive layout); or explicit roots (os.pathsep)
os.environ.setdefault("FF_LITERAL_STRICT", "1")         # stages whose embedded code scans the literal /kaggle/input refuse extra attachments
FF_REPO_GIT = ""                                        # e.g. "https://github.com/nguyennhan2006/FourFusion_FLAG2027.git" (used only without a code bundle)
FF_REPO_REV = ""                                        # the commit to check out (pin it)
print({k: os.environ[k] for k in ("FF_MODE", "FF_TARGET", "FF_USE_SAVED", "FF_ONLY", "FF_FORCE", "FF_SMOKE", "FF_PIP", "FF_TIME_BUDGET_H", "FF_WORK", "FF_DATA", "FF_LITERAL_STRICT")}, flush=True)

In [ ]:
print("[cell] 01 repository: attached code bundle or pinned git clone", flush=True)
import hashlib, json, subprocess, sys
from pathlib import Path
REPO = None
for depth in range(1, 7):
    hits = sorted(Path("/kaggle/input").glob("/".join(["*"] * depth) + "/runner/ffcore.py"))
    hits = [h.parents[1] for h in hits if (h.parents[1] / "configs" / "pipeline_1354.json").exists()]
    if hits:
        REPO = hits[0]
        break
if REPO is None and FF_REPO_GIT:
    REPO = Path("/kaggle/working/ff_repo")
    if not (REPO / "runner" / "ffcore.py").exists():
        subprocess.run(["git", "clone", "-q", FF_REPO_GIT, str(REPO)], check=True)
        if FF_REPO_REV:
            subprocess.run(["git", "-C", str(REPO), "checkout", "-q", FF_REPO_REV], check=True)
assert REPO is not None, "attach the code bundle (runner/build_kaggle_notebook.py --bundle) or set FF_REPO_GIT / FF_REPO_REV"
man = REPO / "ff_code_manifest.json"
if man.exists():
    M = json.loads(man.read_text(encoding="utf-8"))
    bad = [f for f, h in M["files"].items() if hashlib.sha256((REPO / f).read_bytes()).hexdigest() != h]
    assert not bad, f"code bundle altered: {bad[:5]}"
    print("code bundle", REPO, "|", len(M["files"]), "files, sha256 ok | built", M.get("built"), flush=True)
else:
    print("repository", REPO, flush=True)
os.environ["FF_REPO"] = str(REPO)
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

In [ ]:
print("[cell] 02 machine and secrets", flush=True)
import shutil
sys.path.insert(0, str(REPO / "runner"))
import ffcore
got = ffcore.load_kaggle_secrets()
print("python", sys.version.split()[0], "| GPU", ffcore.has_gpu(), "| secrets loaded:", got or "none", flush=True)
roots = ffcore.Config().data_roots                      # FF_DATA=auto: attached Drive-layout folders under /kaggle/input
print("FF_DATA roots:", [str(r) for r in roots] or "none found (mode (c) needs the 05_scores files attached; see the note above)", flush=True)
for r in roots:
    print("   ", r, "->", [d for d in ffcore.DATA_LAYOUT_DIRS if (r / d).is_dir()], flush=True)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"])

In [ ]:
print("[cell] 03 plan", flush=True)
def ffrun(*args):
    p = subprocess.Popen([sys.executable, "-u", str(REPO / "runner" / "ffrun.py"), *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=os.environ.copy())
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()
ffrun("plan")

In [ ]:
print("[cell] 04 run the plan", flush=True)
rc = ffrun("run")
print("ffrun run exit code", rc, flush=True)

In [ ]:
print("[cell] 05 status, final zip, verification", flush=True)
ffrun("status")
work = Path(os.environ["FF_WORK"])
zips = sorted(work.glob("art/notebooks/ff/final_blend/**/xjf_r10_trsig.zip"))
if zips:
    shutil.copy2(zips[0], "/kaggle/working/xjf_r10_trsig.zip")
    ffrun("verify", "zip", "/kaggle/working/xjf_r10_trsig.zip", "--expected", "xjf_r10_trsig")
else:
    print("final zip not produced in this session (see the status above: deferred / blocked stages)", flush=True)
assert rc in (0, 3), f"the run stopped (exit code {rc}); see the log above and FF_WORK/logs/"